In [ ]:
import cv2
video_cap = cv2.VideoCapture(0)
while True:
    ret,video_data = video_cap.read()
    cv2.imshow("Recognition",video_data)
    if cv2.waitKey(10) == ord("e"):
        break
video_cap.release()
    



In [ ]:
import cv2

# Load the pre-trained Haar Cascade Classifier for face detection
face_cascade = cv2.CascadeClassifier(
    "C:/Users/utkar/AppData/Roaming/Python/Python313/site-packages/cv2/data/haarcascade_frontalface_default.xml"
)

# Start capturing video from the webcam (0 is the default webcam)
video_cap = cv2.VideoCapture(0)

# Function to draw angular rectangle (corner lines)
def draw_angular_box(img, x, y, w, h, color=(255, 255, 255), thickness=2, length=20):
    # Top-left corner
    cv2.line(img, (x, y), (x + length, y), color, thickness)
    cv2.line(img, (x, y), (x, y + length), color, thickness)
    # Top-right corner
    cv2.line(img, (x + w, y), (x + w - length, y), color, thickness)
    cv2.line(img, (x + w, y), (x + w, y + length), color, thickness)
    # Bottom-left corner
    cv2.line(img, (x, y + h), (x, y + h - length), color, thickness)
    cv2.line(img, (x, y + h), (x + length, y + h), color, thickness)
    # Bottom-right corner
    cv2.line(img, (x + w, y + h), (x + w - length, y + h), color, thickness)
    cv2.line(img, (x + w, y + h), (x + w, y + h - length), color, thickness)

# Real-time video processing loop
while True:
    ret, frame = video_cap.read()

    # Convert frame to grayscale (better for face detection)
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray_frame,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30),
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    # Draw angular box for each face detected
    for (x, y, w, h) in faces:
        draw_angular_box(frame, x, y, w, h)

    # Display the frame
    cv2.imshow("Face Recognition - Press 'e' or close window to exit", frame)

    # Check if 'e' key is pressed or window is closed (X button)
    if cv2.waitKey(10) == ord("e") or cv2.getWindowProperty("Face Recognition - Press 'e' or close window to exit", cv2.WND_PROP_VISIBLE) < 1:
        break

# Release the video capture and close the window
video_cap.release()
cv2.destroyAllWindows()


In [ ]:

import cv2
import mediapipe as mp

# Load Haar cascade for face detection
face_cascade = cv2.CascadeClassifier(
    "C:/Users/utkar/AppData/Roaming/Python/Python313/site-packages/cv2/data/haarcascade_frontalface_default.xml"
)

# MediaPipe hand detection setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

# Start webcam
video_cap = cv2.VideoCapture(0)

# Function to draw angular box
def draw_angular_box(img, x, y, w, h, color=(255, 255, 255), thickness=2, length=20):
    # Top-left
    cv2.line(img, (x, y), (x + length, y), color, thickness)
    cv2.line(img, (x, y), (x, y + length), color, thickness)
    # Top-right
    cv2.line(img, (x + w, y), (x + w - length, y), color, thickness)
    cv2.line(img, (x + w, y), (x + w, y + length), color, thickness)
    # Bottom-left
    cv2.line(img, (x, y + h), (x, y + h - length), color, thickness)
    cv2.line(img, (x, y + h), (x + length, y + h), color, thickness)
    # Bottom-right
    cv2.line(img, (x + w, y + h), (x + w - length, y + h), color, thickness)
    cv2.line(img, (x + w, y + h), (x + w, y + h - length), color, thickness)

# Loop for real-time detection
while True:
    ret, frame = video_cap.read()
    if not ret:
        break

    # Flip for mirror view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect hand
    result = hands.process(rgb_frame)

    palm_open = False

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            landmarks = hand_landmarks.landmark

            # Detect open fingers
            tip_ids = [4, 8, 12, 16, 20]
            fingers = []

            # Thumb
            if landmarks[4].x > landmarks[3].x:
                fingers.append(1)
            else:
                fingers.append(0)

            # Other 4 fingers
            for tip in tip_ids[1:]:
                if landmarks[tip].y < landmarks[tip - 2].y:
                    fingers.append(1)
                else:
                    fingers.append(0)

            # If all fingers are open, treat as palm open
            if sum(fingers) >= 5:
                palm_open = True
                break

    # Face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30), flags=cv2.CASCADE_SCALE_IMAGE
    )

    for (x, y, w, h) in faces:
        draw_angular_box(frame, x, y, w, h)

    # Exit if palm open
    if palm_open:
        cv2.putText(frame, "Palm Detected - Face Recognition Closed", (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.imshow("Face Recognition - Palm to Close", frame)
        cv2.waitKey(1000)
        break

    # Display info
    cv2.putText(frame, "Show Palm or Press 'e' or Click X to Exit", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 2)

    cv2.imshow("Face Recognition - Palm to Close", frame)

    # Close on 'e' or window X button
    if cv2.waitKey(10) == ord('e') or cv2.getWindowProperty("Face Recognition - Palm to Close", cv2.WND_PROP_VISIBLE) < 1:
        break

# Cleanup
video_cap.release()
cv2.destroyAllWindows()



Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement mediapipe (from versions: none)
ERROR: No matching distribution found for mediapipe


ModuleNotFoundError: No module named 'mediapipe'